In [ ]:
from pathlib import Path
import numpy as np 
import tifffile as tf
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import scipy.cluster.hierarchy as sch 
import matplotlib.pyplot as plt
!pip install matplotlib_scalebar
from matplotlib_scalebar.scalebar import ScaleBar
import seaborn as sns 
import matplotlib
import pickle 

In [ ]:
key  = "" # sample identifier

In [ ]:
pixel_clustering_path = r"" # path to pixel clustering results

In [ ]:
with open(pixel_clustering_path + '/cluster_color_dict.pickle', 'rb') as f: 
    cluster_color_dict = pickle.load(f)

In [ ]:
import matplotlib.patches as mpatches
fig, ax = plt.subplots(1)

i = 0
patches = [
    mpatches.Patch(color=color, label=label)
    for label, color in zip(list(cluster_color_dict.keys()), list(cluster_color_dict.values()))]
ax.legend(patches, list(cluster_color_dict.keys()), loc='center', frameon=False, ncol=int(len(list(cluster_color_dict.keys()))))
ax.axis('off')
i+=1
plt.show()

In [ ]:
non_empty_pixels = pd.read_csv(pixel_clustering_path + '/non_empty_pixels.csv').drop(columns=['Unnamed: 0'])

In [ ]:
non_empty_pixels_som_clusters = pd.read_csv(pixel_clustering_path + '/non_empty_pixels_som_clusters.csv').drop(columns=['Unnamed: 0'])

In [ ]:
sns.set(font_scale=2) 
sns.clustermap(non_empty_pixels_som_clusters.groupby('pixel_som_cluster').median().drop(columns=['X', 'Y']).transpose(), figsize=(20, 10), yticklabels=True, row_cluster=False, cbar_kws={'location': 'right'}, cmap='magma')

In [ ]:
all_color_channels = ['HE_R', 'HE_G', 'HE_B', 'PAS_R', 'PAS_G', 'PAS_B', 'T_R', 'T_G', 'T_B'] 

In [ ]:
non_empty_pixels_unscaled_som_clusters = non_empty_pixels_som_clusters.drop(
    columns=['HE_R', 'HE_G', 'HE_B', 'PAS_R', 'PAS_G', 'PAS_B', 'T_R', 'T_G', 'T_B'])

In [ ]:
for channel in all_color_channels: 
    non_empty_pixels_unscaled_som_clusters[channel + '_no_scale'] = list(non_empty_pixels[channel])

In [ ]:
stains = ['HE', 'PAS', 'T']

In [ ]:
non_empty_pixels_unscaled_som_clusters_statistics = non_empty_pixels_unscaled_som_clusters.groupby('pixel_som_cluster').mean()

In [ ]:
clusters = [5, 8, 9, 4, 6, 12, 3, 1, 2, 7, 10, 11, 16, 13, 14, 15]

In [ ]:
cluster_center_color_rgb_dict = {}
for j in range(len(stains)):
    cluster_center_color_rgb_dict[stains[j]] = {}
    for i in clusters:
        color = (list(non_empty_pixels_unscaled_som_clusters_statistics[stains[j]+'_R_no_scale'])[i-1], list(non_empty_pixels_unscaled_som_clusters_statistics[stains[j]+'_G_no_scale'])[i-1], list(non_empty_pixels_unscaled_som_clusters_statistics[stains[j]+'_B_no_scale'])[i-1])
        cluster_center_color_rgb_dict[stains[j]][i] = color


In [ ]:
import colorsys

In [ ]:
cluster_center_color_hsv_dict = {}
for j in range(len(stains)):
    cluster_center_color_hsv_dict[stains[j]] = {}
    for i in clusters:
        color = cluster_center_color_rgb_dict[stains[j]][i]
        cluster_center_color_hsv_dict[stains[j]][i] = colorsys.rgb_to_hsv(color[0]/255, color[1]/255, color[2]/255)


In [ ]:
if('PAS' in stains): 
    fig, ax = plt.subplots(1, 16)
    fig.set_figheight(2)
    fig.set_figwidth(10)

    stain = 'PAS'
    k = 0
    for i in clusters:
        color = cluster_center_color_hsv_dict[stain][i]  
        bar = np.full((10, 5, 1), color[1])
        ax[k].imshow(bar, cmap='magma', vmin=0, vmax=1)
        k+=1 

            
    for a in ax.ravel():
        a.set_axis_off()

    norm = matplotlib.colors.Normalize(vmin=0, vmax=1)
    sm = matplotlib.cm.ScalarMappable(norm=norm, cmap="magma")
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=ax,
        orientation="vertical",
        fraction=0.1, 
        aspect=10,
        pad=0.02
    )


    cbar.set_ticks([1.0, 0.5, 0.0])
    cbar.set_ticklabels(["1.0", "0.5", "0.0"])

    cbar.ax.tick_params(length=0)  
    cbar.outline.set_visible(False) 

    plt.show()


In [ ]:
fig, ax = plt.subplots(1, 16)
fig.set_figheight(2)
fig.set_figwidth(10)

stain = 'T'
k = 0
for i in clusters:
    color = cluster_center_color_hsv_dict[stain][i]  
    bar = np.full((10, 5, 1), color[0])
    ax[k].imshow(bar, cmap='hsv', vmin=0, vmax=1)
    k+=1 

        
for a in ax.ravel():
    a.set_axis_off()


norm = matplotlib.colors.Normalize(vmin=0, vmax=1)
sm = matplotlib.cm.ScalarMappable(norm=norm, cmap="hsv")
sm.set_array([])

cbar = fig.colorbar(
    sm,
    ax=ax,
    orientation="vertical",
    fraction=0.1, 
    aspect=10,
    pad=0.02
)


cbar.set_ticks([1.0, 0.5, 0.0])
cbar.set_ticklabels(["1.0", "0.5", "0.0"])

cbar.ax.tick_params(length=0)  
cbar.outline.set_visible(False) 

plt.show()

In [ ]:
fig, ax = plt.subplots(1, 16)
fig.set_figheight(2)
fig.set_figwidth(10)

stain = 'T'
k = 0
for i in clusters:
    color = cluster_center_color_hsv_dict[stain][i]  
    bar = np.full((10, 5, 1), color[1])
    ax[k].imshow(bar, cmap='magma', vmin=0, vmax=1)
    k+=1 

        
for a in ax.ravel():
    a.set_axis_off()


norm = matplotlib.colors.Normalize(vmin=0, vmax=1)
sm = matplotlib.cm.ScalarMappable(norm=norm, cmap="magma")
sm.set_array([])

cbar = fig.colorbar(
    sm,
    ax=ax,
    orientation="vertical",
    fraction=0.1, 
    aspect=10,
    pad=0.02
)


cbar.set_ticks([1.0, 0.5, 0.0])
cbar.set_ticklabels(["1.0", "0.5", "0.0"])

cbar.ax.tick_params(length=0)  
cbar.outline.set_visible(False) 

plt.show()

In [ ]:
fig, ax = plt.subplots(3, 16)
fig.set_figheight(2)
fig.set_figwidth(10)

k = 0
for i in clusters:
    for j in range(len(stains)):
        color = (list(non_empty_pixels_unscaled_som_clusters_statistics[stains[j]+'_R_no_scale'])[i-1], list(non_empty_pixels_unscaled_som_clusters_statistics[stains[j]+'_G_no_scale'])[i-1], list(non_empty_pixels_unscaled_som_clusters_statistics[stains[j]+'_B_no_scale'])[i-1])
        bar = np.full((10, 5, 3), (int(color[0]), int(color[1]), int(color[2])))
        ax[j, k].imshow(bar)
    k+=1 
        
for a in ax.ravel():
    a.set_axis_off()